# SIRCA-RAG: Semi-Autonomous RAG for Peruvian Medicinal Plant Knowledge
**SimBig / WAIMLAp 2026**

This notebook demonstrates the complete SIRCA-RAG pipeline:
1. **Hybrid Retrieval** — BGE-M3 dense + BM25 sparse + Cross-Encoder reranking
2. **CRAG Evaluation** — Corrective RAG with accept/refine/web_search decisions
3. **Grounded Generation** — Chain-of-Thought protocol for zero hallucination
4. **Evaluation Metrics** — BERTScore, Faithfulness, Entity Recall, and more

---

## 0. Setup

In [ ]:
import sys, os, json, time
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('.'))

from config.settings import TARGET_SPECIES, EMBEDDING_MODEL, CROSS_ENCODER_MODEL
print(f'Target Species: {len(TARGET_SPECIES)}')
print(f'Embedding Model: {EMBEDDING_MODEL}')
print(f'Cross-Encoder: {CROSS_ENCODER_MODEL}')

## 1. Data Overview
Corpus collected from PubMed, CrossRef, PeruNPDB, GBIF, WFO, and SciELO.

In [ ]:
from pathlib import Path
import json

processed = Path('data/processed')
raw = Path('data/raw')

# Count processed chunks
chunk_files = list(processed.glob('*.json'))
total_chunks = 0
sources = {}
for f in chunk_files:
    data = json.loads(f.read_text(encoding='utf-8'))
    items = data if isinstance(data, list) else [data]
    total_chunks += len(items)
    for item in items:
        src = item.get('source', item.get('metadata', {}).get('source', 'unknown'))
        sources[src] = sources.get(src, 0) + 1

print(f'Processed chunk files: {len(chunk_files)}')
print(f'Total chunks: {total_chunks}')
print(f'\nSources breakdown:')
for src, count in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src}: {count}')

## 2. Hybrid Retrieval Pipeline
BGE-M3 (1024d) dense embeddings + BM25 sparse index, fused via Reciprocal Rank Fusion (RRF), then reranked with a cross-encoder.

In [ ]:
from retrieval.hybrid import HybridRetriever

retriever = HybridRetriever()
retriever.load()

print(f'FAISS index: {retriever._faiss_index.ntotal} vectors (dim={retriever._faiss_index.d})')
print(f'BM25 index: {len(retriever._bm25_corpus)} documents')
print(f'Hybrid alpha: {retriever.alpha} (dense={retriever.alpha:.0%}, sparse={1-retriever.alpha:.0%})')

In [ ]:
# Test retrieval — English query
query_en = "What are the main alkaloids in Uncaria tomentosa?"
start = time.time()
results_en = retriever.retrieve(query_en, rerank_top_k=5)
elapsed = time.time() - start

print(f'Query: {query_en}')
print(f'Retrieved {len(results_en)} docs in {elapsed:.2f}s\n')
for i, r in enumerate(results_en):
    title = r.get('metadata', {}).get('title', 'N/A')[:80]
    score = r.get('rerank_score', r.get('score', 0))
    print(f'  [{i+1}] (score={score:.4f}) {title}')

In [ ]:
# Test retrieval — Spanish query
query_es = "Propiedades medicinales de la maca para la fertilidad"
start = time.time()
results_es = retriever.retrieve(query_es, rerank_top_k=5)
elapsed = time.time() - start

print(f'Query: {query_es}')
print(f'Retrieved {len(results_es)} docs in {elapsed:.2f}s\n')
for i, r in enumerate(results_es):
    title = r.get('metadata', {}).get('title', 'N/A')[:80]
    score = r.get('rerank_score', r.get('score', 0))
    print(f'  [{i+1}] (score={score:.4f}) {title}')

## 3. Query Classification
Rule-based classifier that categorizes queries as `factual`, `exploratory`, or `comparative` and adjusts hybrid retrieval weights.

In [ ]:
from agent.query_classifier import classify_query

test_queries = [
    "What are the main alkaloids in Uncaria tomentosa?",
    "Propiedades medicinales de la maca para la fertilidad",
    "Compare antioxidant activity of Physalis peruviana and yacon",
    "How does Erythroxylum coca differ from cocaine pharmacologically?",
]

for q in test_queries:
    c = classify_query(q)
    print(f'  [{c.category:>12}] (conf={c.confidence:.2f}, alpha={c.alpha_override}) {q[:60]}...')

## 4. CRAG Evaluation
Corrective RAG evaluator that decides whether to **accept** retrieved docs, **refine** the query, or **fall back to web search**.

In [ ]:
from agent.crag_evaluator import CRAGEvaluator

evaluator = CRAGEvaluator()

# Evaluate the English retrieval results
decision = evaluator.evaluate(query_en, results_en)
print(f'Query: {query_en}')
print(f'  Action: {decision.action}')
print(f'  Confidence: {decision.confidence:.4f}')
print(f'  Relevant indices: {decision.relevant_indices}')
print(f'  Reason: {decision.reason}')

## 5. Full Agent Pipeline (End-to-End)
The `SIRCAAgent` orchestrates the full CRAG pipeline: classify → retrieve → evaluate → [accept|refine|web_search] → generate.

In [ ]:
from agent.graph import SIRCAAgent

# Use template backend for fast demo (replace with 'deepseek' for full quality)
agent = SIRCAAgent(retriever=retriever, generator_backend='template')

start = time.time()
result = agent.run(query_en)
elapsed = time.time() - start

gen = result.get('generation', {})
crag = result.get('crag_decision', {})
classification = result.get('classification', {})

print(f'Query: {query_en}')
print(f'Category: {classification.get("category")} | CRAG: {crag.get("action")} | Time: {elapsed:.1f}s')
print(f'Model: {gen.get("model")} | Tokens: {gen.get("tokens_generated")}')
print(f'\n--- Answer ---')
print(gen.get('answer', 'No answer')[:500])
print(f'\n--- Citations ({len(result.get("citations", []))}) ---')
for c in result.get('citations', []):
    print(f'  [{c["index"]}] {c.get("title", "N/A")[:70]} | {c.get("species", [])}')
print(f'\n--- Trace ---')
for step in result.get('trace', []):
    dur = step.get('duration_ms', 0)
    print(f'  {step["node"]} ({dur}ms)')

In [ ]:
# Spanish query test
start = time.time()
result_es = agent.run(query_es)
elapsed = time.time() - start

gen_es = result_es.get('generation', {})
crag_es = result_es.get('crag_decision', {})

print(f'Query: {query_es}')
print(f'CRAG: {crag_es.get("action")} | Time: {elapsed:.1f}s')
print(f'\n--- Answer ---')
print(gen_es.get('answer', 'No answer')[:500])

## 6. Evaluation Metrics
Full benchmark on 8 curated test cases covering all 8 target species.

### Metrics computed:
| Metric | Description |
|--------|-------------|
| BERTScore F1 | Semantic similarity (roberta-large) |
| Semantic Similarity | Cross-encoder relevance score |
| Context Precision | Precision of retrieved relevant docs |
| Context Recall | Recall of retrieved relevant docs |
| MRR | Mean Reciprocal Rank |
| NDCG@5 | Normalized Discounted Cumulative Gain |
| Entity Recall | Scientific entity coverage |
| Faithfulness | Grounding to source context |
| Answer Relevancy | Query-answer semantic alignment |

In [ ]:
# Load pre-computed results (DeepSeek backend)
eval_path = Path('data/evaluation_deepseek.json')
if eval_path.exists():
    eval_results = json.loads(eval_path.read_text())
    print('Pre-computed Evaluation Results (DeepSeek backend)')
    print('=' * 55)
    print(f'{"Metric":<30} {"Score":>10} {"Target":>10}')
    print('-' * 55)
    targets = {
        'bertscore_f1': '≥ 0.90',
        'semantic_similarity': '—',
        'context_precision': '—',
        'context_recall': '—',
        'mrr': '—',
        'ndcg@5': '—',
        'entity_recall': '—',
        'faithfulness': '≥ 0.80',
        'answer_relevancy': '—',
    }
    for metric, score in eval_results.items():
        if isinstance(score, (int, float)):
            target = targets.get(metric, '—')
            status = '✓' if target != '—' and score >= float(target.split()[-1]) else ' '
            print(f'  {metric:<28} {score:>10.4f} {target:>10} {status}')
    print('=' * 55)
else:
    print('No pre-computed results found. Run: python -m evaluation.benchmark --backend deepseek')

In [ ]:
# Run live evaluation with template backend (fast, lower quality)
from evaluation.benchmark import run_benchmark

print('Running live benchmark with template backend...\n')
live_results = run_benchmark(backend='template')

## 7. Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                      SIRCA-RAG Pipeline                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Query ──► Classify ──► Retrieve ──► Evaluate ──► Generate  │
│            (rule-     (BGE-M3 +    (CRAG       (CoT        │
│             based)     BM25 +       evaluator)   grounded)  │
│                        rerank)        │                     │
│                          ▲            │                     │
│                          │       ┌────┴────┐                │
│                          │       │ accept  │                │
│                     Refine◄──────┤ refine  │                │
│                                  │web_search│               │
│                                  └─────────┘                │
├─────────────────────────────────────────────────────────────┤
│  Data Sources: PubMed, CrossRef, PeruNPDB, GBIF, WFO       │
│  Vectorstore: FAISS (3040 vectors, 1024d BGE-M3)            │
│  Generators: DeepSeek V4 Flash, Ollama (Qwen3.5), Template  │
│  Species: 8 Peruvian medicinal plants                       │
└─────────────────────────────────────────────────────────────┘
```

## 8. Web Service (MVP)
FastAPI service at `localhost:8000` with:
- `GET /` — Interactive dark-themed frontend
- `GET /api/health` — System status, backend availability, vectorstore size
- `POST /api/query` — Full pipeline execution with citations and trace
- `GET /api/species` — List of target species

Start with:
```bash
python -m web.app
```